# Fine-tuning sur GPU Colab — MS COCO multi-label

Le fine-tuning complet est hors de portee du CPU de developpement : une epoque
ResNet18 a 224 px y prend environ 50 minutes, et une epoque ResNet50 2 h 34
(mesures de `scripts/benchmark_speed.py`). Sur un GPU Colab (T4), comptez
quelques minutes par epoque.

Ce notebook execute **le meme code** que le depot local : les modules de
`src/coco_mlc/` sont clones, aucune logique n'est reecrite ici. Les resultats
sont donc directement comparables a ceux obtenus en local.

**Verifier d'abord :** Execution > Modifier le type d'execution > GPU.

## 1. Verifier le GPU attribue

In [ ]:
import torch

print("torch", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {props.name}, {props.total_memory / 1e9:.1f} Go")
else:
    raise SystemExit("Aucun GPU : Execution > Modifier le type d'execution > GPU")

## 2. Recuperer le code et les donnees

Le dataset fait 2,2 Go. Le deposer **une fois** sur Google Drive sous forme
d'archive, puis la decompresser dans le disque local de la machine Colab
(`/content`) a chaque session : lire 65 000 petits fichiers directement depuis
Drive est beaucoup plus lent que depuis le disque local.

Creation de l'archive en local, avant de l'envoyer sur Drive :

```bash
cd /chemin/vers/le/parent/de/ms-coco
tar -czf ms-coco.tar.gz ms-coco
```

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e

# Adapter le chemin de l'archive sur Drive.
ARCHIVE=/content/drive/MyDrive/ms-coco.tar.gz

if [ ! -d /content/ms-coco ]; then
  echo "Decompression de $ARCHIVE ..."
  tar -xzf "$ARCHIVE" -C /content
fi

echo "train : $(ls /content/ms-coco/images/train | wc -l) images"
echo "test  : $(ls /content/ms-coco/images/test | wc -l) images"
echo "labels: $(ls /content/ms-coco/labels/train | wc -l) fichiers"

In [ ]:
%%bash
set -e
cd /content
if [ ! -d IAV_MS_COCO ]; then
  git clone https://github.com/proche0/IAV_MS_COCO.git
fi
cd IAV_MS_COCO
git pull --ff-only || true
git log --oneline -1

In [ ]:
import os

# torch et torchvision sont deja installes sur Colab : ne pas les reinstaller,
# au risque de casser la correspondance avec la version de CUDA.
os.environ["MSCOCO_ROOT"] = "/content/ms-coco"
os.chdir("/content/IAV_MS_COCO")

!pip install -q tqdm torchinfo
!python3 tests/test_metrics_parity.py

## 3. Verifier le pipeline avant de lancer un entrainement long

Une epoque sur 2 000 images suffit a detecter une erreur de chemin, de forme de
tenseur ou de fonction de cout. Toujours faire ce test avant un entrainement de
plusieurs heures.

In [ ]:
!python3 scripts/train.py --model mobilenet_v3_small --max-images 2000 --epochs 1 --num-workers 2

## 4. Entrainements complets

Une seule variable change entre les runs, pour que les ecarts soient
interpretables. Le `--loss asl` (Asymmetric Loss) est la reference sur MS COCO
multi-label ; le comparer a `bce` quantifie l'apport du traitement du
desequilibre.

`--amp` active la precision mixte, qui divise environ par deux le temps par
epoque sur GPU.

In [ ]:
# Baseline fine-tunee, a comparer a la tete sur features obtenue en local.
!python3 scripts/train.py --model resnet18 --epochs 10 --loss bce \
    --batch-size 128 --lr 1e-4 --amp --num-workers 2 --tensorboard

In [ ]:
# Meme modele, fonction de cout adaptee au desequilibre.
!python3 scripts/train.py --model resnet18 --epochs 10 --loss asl \
    --batch-size 128 --lr 1e-4 --amp --num-workers 2

In [ ]:
# Backbone plus performant (poids IMAGENET1K_V2, 80,9 % top-1).
!python3 scripts/train.py --model resnet50 --epochs 10 --loss asl \
    --batch-size 96 --lr 1e-4 --amp --num-workers 2

In [ ]:
# Convolution moderne contre transformer, a cout de calcul quasi identique
# (4,46 contre 4,49 GFLOPS).
!python3 scripts/train.py --model convnext_tiny --epochs 10 --loss asl \
    --batch-size 64 --lr 5e-5 --amp --num-workers 2
!python3 scripts/train.py --model swin_t --epochs 10 --loss asl \
    --batch-size 64 --lr 5e-5 --amp --num-workers 2

## 5. Comparer, calibrer, predire

In [ ]:
import pandas as pd

experiments = pd.read_csv("outputs/experiments.csv")
columns = ["backbone", "strategy", "loss", "val_f1", "val_precision",
           "val_recall", "val_mAP", "best_epoch", "seconds"]
experiments[[c for c in columns if c in experiments]].sort_values("val_f1", ascending=False)

In [ ]:
# Remplacer par le meilleur checkpoint du tableau ci-dessus.
CHECKPOINT = "outputs/model_resnet50_asl.pth"

!python3 scripts/tune_thresholds.py --checkpoint {CHECKPOINT} --save
!python3 scripts/predict.py --checkpoint {CHECKPOINT} --submit-copy

## 6. Recuperer les resultats

Les machines Colab sont effacees a la fin de la session : copier les
checkpoints, les historiques et le JSON de soumission sur Drive avant de
fermer.

In [ ]:
%%bash
DEST=/content/drive/MyDrive/iav_ms_coco_resultats
mkdir -p "$DEST"
cp -v outputs/*.csv outputs/*.json outputs/*.pth "$DEST" 2>/dev/null
cp -rv runs "$DEST" 2>/dev/null
ls -lh "$DEST" | tail -20